In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import sklearn

**Crypto price prediction depends on two main factors: historical time series data and real-world market sentiment. Unlike stock price prediction, where price history often carries most of the signal, crypto is strongly influenced by traders' reactions to political, economic and financial events worldwide. Our dataset provides only the price side, namely open, high, low, close and daily variation, so we must separately obtain a sentiment score for each day in the historical window we are studying. Merging these daily scores with the OHLC data by date gives the model both price behavior and market mood as inputs.**

In [2]:
pd.options.display.float_format = '{:.2f}'.format
pd.options.display.max_rows = 100
pd.options.display.max_columns = 25

In [3]:
# Load data from source

data = pd.read_csv(r"C:\Users\ACER\OneDrive\Desktop\ML Notes and colab Notebooks\Data Science Projects\Cryptoprice-prediction\notebook\data\Crypto_dataset.csv")

In [4]:
df = data.copy()

In [5]:
df

,Unnamed: 0,open,high,low,close,volume,marketCap,timestamp,crypto_name,date
0,0,112.90,118.80,107.14,115.91,0.00,1288693175.50,2013-05-05T23:59:59.999Z,Bitcoin,2013-05-05
1,1,3.49,3.69,3.35,3.59,0.00,62298185.43,2013-05-05T23:59:59.999Z,Litecoin,2013-05-05
2,2,115.98,124.66,106.64,112.30,0.00,1249023060.00,2013-05-06T23:59:59.999Z,Bitcoin,2013-05-06
3,3,3.59,3.78,3.12,3.37,0.00,58594361.23,2013-05-06T23:59:59.999Z,Litecoin,2013-05-06
4,4,112.25,113.44,97.70,111.50,0.00,1240593600.00,2013-05-07T23:59:59.999Z,Bitcoin,2013-05-07
...,...,...,...,...,...,...,...,...,...,...
72941,72941,0.02,0.02,0.02,0.02,40401338.08,1652957186.63,2022-10-23T23:59:59.999Z,VeChain,2022-10-23
72942,72942,1.47,1.53,1.44,1.52,28443510.43,1572824923.08,2022-10-23T23:59:59.999Z,Flow,2022-10-23
72943,72943,4.95,5.15,4.95,5.12,106949683.62,1559551358.49,2022-10-23T23:59:59.999Z,Filecoin,2022-10-23
72944,72944,0.00,0.00,0.00,0.00,214326817.63,1576291167.45,2022-10-23T23:59:59.999Z,Terra Classic,2022-10-23


In [6]:
df['crypto_name'].value_counts()


crypto_name
Bitcoin                  3248
Litecoin                 3248
XRP                      3157
Dogecoin                 3024
Monero                   2866
Stellar                  2791
Tether                   2582
Ethereum                 2424
Ethereum Classic         2072
Basic Attention Token    1760
EOS                      1730
Bitcoin Cash             1708
BNB                      1706
TRON                     1656
Decentraland             1652
Chainlink                1649
Cardano                  1638
Maker                    1605
Filecoin                 1565
Theta Network            1530
Huobi Token              1513
Ravencoin                1478
Tezos                    1365
VeChain                  1332
Quant                    1325
USD Coin                 1266
Cronos                   1199
Wrapped Bitcoin          1152
Cosmos                   1109
Polygon                  1064
OKB                      1062
UNUS SED LEO             1041
Algorand                 101

# Feature Engeenering

In [7]:
df = df.drop(columns=["Unnamed: 0",'timestamp'], axis = 1)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72946 entries, 0 to 72945
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         72946 non-null  float64
 1   high         72946 non-null  float64
 2   low          72946 non-null  float64
 3   close        72946 non-null  float64
 4   volume       72946 non-null  float64
 5   marketCap    72946 non-null  float64
 6   crypto_name  72946 non-null  object 
 7   date         72946 non-null  object 
dtypes: float64(6), object(2)
memory usage: 4.5+ MB


In [9]:
df.date = df.date.str.replace('-','/')
df.date = pd.to_datetime(df['date'])

df['asset_id'] = df["crypto_name"].astype('category').cat.codes

In [10]:
#To see actual crypto names 
#df["crypto_name"].astype('category').cat.categories

In [11]:
dict = dict(zip(df['asset_id'].unique(), df['crypto_name'].unique()))

In [12]:
print(dict)

{np.int8(8): 'Bitcoin', np.int8(30): 'Litecoin', np.int8(55): 'XRP', np.int8(19): 'Dogecoin', np.int8(32): 'Monero', np.int8(41): 'Stellar', np.int8(45): 'Tether', np.int8(22): 'Ethereum', np.int8(23): 'Ethereum Classic', np.int8(31): 'Maker', np.int8(6): 'Basic Attention Token', np.int8(20): 'EOS', np.int8(9): 'Bitcoin Cash', np.int8(5): 'BNB', np.int8(43): 'TRON', np.int8(18): 'Decentraland', np.int8(13): 'Chainlink', np.int8(10): 'Cardano', np.int8(25): 'Filecoin', np.int8(48): 'Theta Network', np.int8(28): 'Huobi Token', np.int8(38): 'Ravencoin', np.int8(46): 'Tezos', np.int8(53): 'VeChain', np.int8(37): 'Quant', np.int8(51): 'USD Coin', np.int8(16): 'Cronos', np.int8(54): 'Wrapped Bitcoin', np.int8(15): 'Cosmos', np.int8(36): 'Polygon', np.int8(34): 'OKB', np.int8(50): 'UNUS SED LEO', np.int8(1): 'Algorand', np.int8(14): 'Chiliz', np.int8(42): 'THORChain', np.int8(44): 'Terra Classic', np.int8(24): 'FTX Token', np.int8(27): 'Hedera', np.int8(7): 'Binance USD', np.int8(17): 'Dai', 

In [13]:
df['asset_id']

0         8
1        30
2         8
3        30
4         8
         ..
72941    53
72942    26
72943    25
72944    44
72945    55
Name: asset_id, Length: 72946, dtype: int8

In [15]:
df.drop(columns= 'crypto_name', axis = 1)

,open,high,low,close,volume,marketCap,date,asset_id
0,112.90,118.80,107.14,115.91,0.00,1288693175.50,2013-05-05,8
1,3.49,3.69,3.35,3.59,0.00,62298185.43,2013-05-05,30
2,115.98,124.66,106.64,112.30,0.00,1249023060.00,2013-05-06,8
3,3.59,3.78,3.12,3.37,0.00,58594361.23,2013-05-06,30
4,112.25,113.44,97.70,111.50,0.00,1240593600.00,2013-05-07,8
...,...,...,...,...,...,...,...,...
72941,0.02,0.02,0.02,0.02,40401338.08,1652957186.63,2022-10-23,53
72942,1.47,1.53,1.44,1.52,28443510.43,1572824923.08,2022-10-23,26
72943,4.95,5.15,4.95,5.12,106949683.62,1559551358.49,2022-10-23,25
72944,0.00,0.00,0.00,0.00,214326817.63,1576291167.45,2022-10-23,44


In [16]:
import xgboost